# 06 预训练（Pre-training） / Pre-training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiYuBY/use_token_to_learn/blob/main/docs/06-pretraining/06-pretraining.ipynb)

[📖 讲解 / Notes](https://baiyuby.github.io/use_token_to_learn/06-pretraining/README.html) · [🎮 交互演示 / Playground](https://baiyuby.github.io/use_token_to_learn/06-pretraining/interactive/pretraining.html) · [🧑‍💻 GitHub](https://github.com/BaiYuBY/use_token_to_learn/tree/main/docs/06-pretraining)

这个 notebook 由 `scripts/make_notebooks.py` 根据本章 `code/` 目录下的脚本自动生成，代码与脚本完全一致：每个函数或类是一个单元格，
可以逐个修改、重新运行。最后一个单元格运行本章的演示（带 assert 检查）。

Generated by `scripts/make_notebooks.py` from this chapter's `code/` scripts, with identical code: one cell per function or
class, so you can edit and re-run them one at a time. The last cell runs the chapter's demo (with its assert checks).

In [ ]:
# 环境 / setup: find the repository, or clone it (e.g. in Colab); some chapters import code from earlier ones
# 找到仓库，找不到就克隆一份（例如在 Colab 中）；有些章节会导入前面章节的代码
import subprocess
import sys
from pathlib import Path

CHAPTER = "06-pretraining"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / CHAPTER).is_dir()), None)
if ROOT is None:
    ROOT = Path.cwd() / "use_token_to_learn"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BaiYuBY/use_token_to_learn.git", str(ROOT)], check=True)
CODE = ROOT / "docs" / CHAPTER / "code"
sys.path.insert(0, str(CODE))
print("code:", CODE)

## `data_filters.py`

```text
A toy pre-training data pipeline / 一个玩具版的预训练数据清洗流水线.

Run / 运行:
    python docs/06-pretraining/code/data_filters.py

Real pipelines (e.g. FineWeb, RefinedWeb, Dolma) use the same kinds of steps at a much larger scale:
heuristic quality filters, language identification, deduplication and PII removal.
真实的流水线（如 FineWeb、RefinedWeb、Dolma）用的也是这几类步骤，只是规模大得多。
```

In [ ]:
__file__ = str(CODE / "data_filters.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import hashlib
import re

In [ ]:
THRESHOLDS = {
    "min_words": 12,  # too short to be useful / 太短
    "max_caps_ratio": 0.5,  # SHOUTING CLICKBAIT / 全大写标题党
    "max_symbol_ratio": 0.3,  # menus, markup debris / 菜单、标记残渣
    "min_unique_ratio": 0.4,  # "buy now buy now buy now" / 重复刷屏
    "near_dup_jaccard": 0.6,  # word 3-gram overlap with an earlier document / 与已保留文档的 3-gram 重合度
    "min_stopword_ratio": 0.15,  # English needs some "the", "of", "and" / 英文应有一定比例的功能词
    "min_cjk_ratio": 0.3,  # Chinese characters / 汉字比例
}
STOPWORDS = {"the", "a", "an", "of", "and", "to", "in", "is", "it", "that", "for", "on", "with", "as", "was",
             "are", "be", "by", "this", "from", "or", "at", "which", "can", "their", "they", "we", "you"}
FILTERS = ["length", "language", "caps", "symbols", "repetition", "exact_dedup", "near_dedup", "pii"]
DOCS = [
    ("web-1", "The octopus is a soft-bodied animal with eight arms. It can change the colour and texture of its "
              "skin in a fraction of a second, which helps it hide from predators and surprise its prey."),
    ("web-2", "The octopus is a soft-bodied animal with eight arms. It can change the colour and texture of its "
              "skin in a fraction of a second, which helps it hide from predators and surprise its prey."),
    ("web-3", "The octopus is a soft-bodied animal with eight arms. It can change the colour and texture of its "
              "skin in a fraction of a second, which helps it hide from predators and catch its prey."),
    ("web-4", "Click here to continue."),
    ("web-5", "buy now buy now cheap cheap buy now best price buy now cheap buy now best price buy now cheap"),
    ("web-6", "Home | About | Blog | Shop | Login | Register | FAQ | Contact | >> | << | ### | *** | @@ | Terms"),
    ("web-7", "YOU WON'T BELIEVE WHAT HAPPENED NEXT!!! THIS ONE WEIRD TRICK WILL CHANGE YOUR LIFE FOREVER, "
              "CLICK NOW BEFORE IT IS TOO LATE"),
    ("web-8", "La fotosíntesis es el proceso mediante el cual las plantas convierten la luz solar en energía "
              "química, liberando oxígeno como subproducto."),
    ("zh-1", "光合作用是植物利用阳光、水和二氧化碳制造养分的过程。在这个过程中，叶绿体吸收光能，把它转化为化学能，"
             "同时释放出氧气。"),
    ("code-1", "def fibonacci(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n"
               "    return a\n\nprint([fibonacci(i) for i in range(10)])"),
    ("web-9", "For questions about the workshop, write to the organiser at jane.doe@example.com or call "
              "555-123-4567 between nine and five on weekdays, and we will get back to you."),
    ("web-10", "Water boils at a lower temperature on a high mountain because the air pressure is lower, so the "
               "molecules need less energy to escape from the liquid as vapour."),
]
WORD = re.compile(r"[A-Za-zÀ-ÿ0-9']+|[一-鿿]")
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b")
CODE_LINE = re.compile(r"^\s*(def|class|import|from|return|for|if|print)\b", re.M)

In [ ]:
def words(text):
    return WORD.findall(text)

In [ ]:
def language(text):
    """Very simplified language ID / 极简的语言识别: zh, en, code or other."""
    toks = words(text)
    if len(CODE_LINE.findall(text)) >= 2:
        return "code"
    cjk = sum(1 for t in toks if "一" <= t <= "鿿")
    if toks and cjk / len(toks) >= THRESHOLDS["min_cjk_ratio"]:
        return "zh"
    stop = sum(1 for t in toks if t.lower() in STOPWORDS)
    if toks and stop / len(toks) >= THRESHOLDS["min_stopword_ratio"]:
        return "en"
    return "other"

In [ ]:
def shingles(text, n=3):
    toks = [t.lower() for t in words(text)]
    return {tuple(toks[i:i + n]) for i in range(max(1, len(toks) - n + 1))}

In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0

In [ ]:
def quality_reasons(text, enabled):
    """Every heuristic rule this document breaks / 这篇文档违反的所有启发式规则."""
    toks = words(text)
    letters = [c for c in text if c.isalpha()]
    non_space = [c for c in text if not c.isspace()]
    checks = {
        "length": len(toks) < THRESHOLDS["min_words"],
        "language": language(text) == "other",
        "caps": bool(letters) and sum(c.isupper() for c in letters) / len(letters) > THRESHOLDS["max_caps_ratio"],
        "symbols": bool(non_space) and sum(not c.isalnum() for c in non_space) / len(non_space) > THRESHOLDS["max_symbol_ratio"],
        "repetition": bool(toks) and len({t.lower() for t in toks}) / len(toks) < THRESHOLDS["min_unique_ratio"],
    }
    return [name for name, failed in checks.items() if failed and name in enabled]

In [ ]:
def run_pipeline(docs, enabled=FILTERS):
    """Returns [(doc_id, kept, reasons, text)] in input order. Quality rules run first; documents that
    pass them are deduplicated against the documents kept so far.
    按输入顺序返回去留、原因列表和处理后的文本。先做质量过滤，通过的文档再与已保留的文档去重。"""
    out, seen_hashes, kept_shingles = [], set(), []
    for doc_id, text in docs:
        reasons = quality_reasons(text, enabled)
        digest = hashlib.md5(" ".join(text.lower().split()).encode()).hexdigest()
        if not reasons and "exact_dedup" in enabled and digest in seen_hashes:
            reasons = ["exact_dedup"]
        if not reasons and "near_dedup" in enabled and any(
                jaccard(shingles(text), s) >= THRESHOLDS["near_dup_jaccard"] for s in kept_shingles):
            reasons = ["near_dedup"]
        if not reasons:
            seen_hashes.add(digest)
            kept_shingles.append(shingles(text))
            if "pii" in enabled:
                text = PHONE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))
        out.append((doc_id, not reasons, reasons, text))
    return out

In [ ]:
EXPECTED = {"web-1": [], "web-2": ["exact_dedup"], "web-3": ["near_dedup"], "web-4": ["length"],
            "web-5": ["language", "repetition"], "web-6": ["length", "language", "symbols"], "web-7": ["caps"],
            "web-8": ["language"], "zh-1": [], "code-1": [], "web-9": [], "web-10": []}

In [ ]:
def main():
    results = run_pipeline(DOCS)
    for doc_id, kept, reasons, text in results:
        status = "keep" if kept else "drop (" + ", ".join(reasons) + ")"
        print(f"{doc_id:7s} {status:36s} {text[:50]!r}")
        assert reasons == EXPECTED[doc_id], (doc_id, reasons)
    masked = dict((d, t) for d, _, _, t in results)["web-9"]
    assert "<EMAIL>" in masked and "<PHONE>" in masked
    print(f"✓ kept {sum(r[1] for r in results)} of {len(DOCS)} documents; personal data masked")

    no_dedup = dict((d, k) for d, k, _, _ in run_pipeline(DOCS, enabled=["length", "language"]))
    assert no_dedup["web-2"] and no_dedup["web-3"], "without dedup the copies slip through"
    print("✓ without deduplication, the exact and near copies of web-1 are kept as well")

### 运行演示 / Run the demo (`data_filters.py`)

In [ ]:
main()

## `pretrain.py`

```text
Pre-training a small language model in PyTorch / 用 PyTorch 预训练一个小语言模型.

Run / 运行:
    python docs/06-pretraining/code/pretrain.py

Covers / 包含:
    1. next-token prediction, cross-entropy and perplexity     下一个词元预测、交叉熵与困惑度
    2. learning-rate schedules: warmup + cosine, WSD          学习率调度：预热 + 余弦、WSD
    3. AdamW, gradient clipping, train vs validation loss     AdamW、梯度裁剪、训练与验证损失
    4. too little data -> memorization; more data -> generalization
       数据太少只会背诵；数据够多才能泛化
```

In [ ]:
__file__ = str(CODE / "pretrain.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
SUBJECTS = {"cat": "cats", "dog": "dogs", "bird": "birds", "child": "children", "teacher": "teachers",
            "robot": "robots", "farmer": "farmers", "student": "students"}
OBJECTS = {"book": "books", "apple": "apples", "ball": "balls", "letter": "letters", "song": "songs",
           "cat": "cats", "dog": "dogs", "bird": "birds"}
ADJECTIVES = ["small", "big", "happy", "old", "young", "quiet", "clever", "red"]
TRANSITIVE = {"sees": "see", "likes": "like", "finds": "find", "chases": "chase", "reads": "read",
              "writes": "write", "helps": "help"}
INTRANSITIVE = {"sleeps": "sleep", "runs": "run", "sings": "sing", "waits": "wait", "laughs": "laugh"}
PLACES = ["park", "house", "river", "school", "garden"]
PREPOSITIONS = ["in", "near", "behind", "under"]
ADVERBS = ["quickly", "slowly", "today", "again"]

In [ ]:
def noun_phrase(rng, nouns, plural):
    det = rng.choice(["the", "some", "many", "two"] if plural else ["the", "a", "every"])
    noun = rng.choice(list(nouns))
    words = [det] + ([rng.choice(ADJECTIVES)] if rng.random() < 0.4 else [])
    words += [nouns[noun] if plural else noun]
    if words[0] == "a" and words[1][0] in "aeiou":
        words[0] = "an"  # "an old dog", "an apple"
    return words

In [ ]:
def verb(rng, table, plural):
    v = rng.choice(list(table))
    return table[v] if plural else v

In [ ]:
def sentence(rng):
    plural = rng.random() < 0.5
    words = noun_phrase(rng, SUBJECTS, plural)
    if rng.random() < 0.3:  # relative clause between subject and verb / 主语和动词之间插入从句
        words += ["that", verb(rng, TRANSITIVE, plural)] + noun_phrase(rng, OBJECTS, rng.random() < 0.5)
    if rng.random() < 0.5:
        words += [verb(rng, INTRANSITIVE, plural)]
    else:
        words += [verb(rng, TRANSITIVE, plural)] + noun_phrase(rng, OBJECTS, rng.random() < 0.5)
    if rng.random() < 0.4:
        words += [rng.choice(PREPOSITIONS), "the", rng.choice(PLACES)]
    if rng.random() < 0.3:
        words += [rng.choice(ADVERBS)]
    return words + ["."]

In [ ]:
def make_corpus(n_sentences, seed):
    rng = random.Random(seed)
    return [w for _ in range(n_sentences) for w in sentence(rng)]

In [ ]:
VOCAB = sorted({w for w in make_corpus(5000, seed=12345)} | set(SUBJECTS.values()) | set(OBJECTS.values()))
STOI = {w: i for i, w in enumerate(VOCAB)}

In [ ]:
def encode(words):
    return torch.tensor([STOI[w] for w in words])

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d=64, n_layers=2, n_heads=4, ctx=32):
        super().__init__()
        self.ctx = ctx
        self.embed = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(ctx, d)  # learned positions (GPT-2 style) / 可学习位置编码
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        for p in self.parameters():
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, ids):
        T = ids.size(1)
        x = self.embed(ids) + self.pos(torch.arange(T, device=ids.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=ids.device)
        return self.head(self.norm(self.blocks(x, mask=mask, is_causal=True)))

### Learning-rate schedules / 学习率调度

In [ ]:
def constant_lr(step, total, peak, warmup=0):
    return peak

In [ ]:
def cosine_lr(step, total, peak, warmup, min_ratio=0.1):
    """Linear warmup, then cosine decay to min_ratio * peak / 线性预热，再余弦衰减到 min_ratio * peak."""
    if step < warmup:
        return peak * (step + 1) / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return peak * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress)))

In [ ]:
def wsd_lr(step, total, peak, warmup, decay_frac=0.2, min_ratio=0.0):
    """Warmup-Stable-Decay: warm up, hold the peak, decay linearly at the end / 预热、保持、最后线性衰减."""
    if step < warmup:
        return peak * (step + 1) / warmup
    decay_start = int(total * (1 - decay_frac))
    if step < decay_start:
        return peak
    return peak * (min_ratio + (1 - min_ratio) * (total - step) / max(1, total - decay_start))

In [ ]:
@torch.no_grad()
def evaluate(model, data, ctx, n_batches=8, batch=32, seed=0):
    g = torch.Generator().manual_seed(seed)
    losses = []
    for _ in range(n_batches):
        starts = torch.randint(0, len(data) - ctx - 1, (batch,), generator=g)
        x = torch.stack([data[s:s + ctx] for s in starts])
        y = torch.stack([data[s + 1:s + ctx + 1] for s in starts])
        losses.append(F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item())
    return sum(losses) / len(losses)

In [ ]:
def train(train_data, val_data, steps=600, schedule=cosine_lr, peak_lr=3e-3, warmup=50, batch=32,
          ctx=32, clip=1.0, eval_every=25, seed=0, probe=None):
    """Returns the model and a log of steps, train/val loss, learning rate, gradient norm and,
    if given, probe(model) at every evaluation point.
    返回模型，以及每个评估点的步数、训练/验证损失、学习率、梯度范数，以及可选的 probe(model)。"""
    torch.manual_seed(seed)
    model = GPT(len(VOCAB), ctx=ctx)
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, betas=(0.9, 0.95), weight_decay=0.1)
    g = torch.Generator().manual_seed(seed)
    log = {"step": [], "train": [], "val": [], "lr": [], "grad_norm": [], "probe": []}
    running, norms = [], []
    for step in range(steps + 1):
        if step % eval_every == 0:
            model.eval()
            log["step"].append(step)
            log["train"].append(sum(running) / len(running) if running else evaluate(model, train_data, ctx))
            log["val"].append(evaluate(model, val_data, ctx))
            log["lr"].append(schedule(min(step, steps - 1), steps, peak_lr, warmup))
            log["grad_norm"].append(sum(norms) / len(norms) if norms else 0.0)
            if probe:
                log["probe"].append(probe(model))
            running, norms = [], []
            model.train()
        if step == steps:
            break
        for group in opt.param_groups:
            group["lr"] = schedule(step, steps, peak_lr, warmup)
        starts = torch.randint(0, len(train_data) - ctx - 1, (batch,), generator=g)
        x = torch.stack([train_data[s:s + ctx] for s in starts])
        y = torch.stack([train_data[s + 1:s + ctx + 1] for s in starts])  # targets: shifted by one / 右移一位
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad()
        loss.backward()
        norms.append(nn.utils.clip_grad_norm_(model.parameters(), clip).item())  # clip and record
        opt.step()
        running.append(loss.item())
    return model, log

In [ ]:
SINGULAR_VERBS = [STOI[v] for v in list(TRANSITIVE) + list(INTRANSITIVE)]
PLURAL_VERBS = [STOI[v] for v in list(TRANSITIVE.values()) + list(INTRANSITIVE.values())]

In [ ]:
@torch.no_grad()
def agreement_accuracy(model, n=300, seed=7):
    """Did the model learn subject-verb agreement? / 模型学会主谓一致了吗？
    Prefix: subject + relative clause whose object has the OTHER number, e.g.
    "the cat that chases the dogs ___". The nearest noun is plural, but the verb must be singular.
    Count how often the model puts more probability on the correct verb forms.
    前缀中最近的名词与主语单复数相反，看模型是否仍把更多概率给正确的动词形式。"""
    model.eval()
    rng, correct = random.Random(seed), 0
    for _ in range(n):
        plural = rng.random() < 0.5
        words = noun_phrase(rng, SUBJECTS, plural) + ["that", verb(rng, TRANSITIVE, plural)]
        words += noun_phrase(rng, OBJECTS, not plural)
        probs = model(encode(words)[None])[0, -1].softmax(-1)
        p_sing, p_plur = probs[SINGULAR_VERBS].sum(), probs[PLURAL_VERBS].sum()
        correct += int((p_plur > p_sing) if plural else (p_sing > p_plur))
    return correct / n

In [ ]:
@torch.no_grad()
def next_word_probs(model, words, k=5):
    """Top-k predictions after each prefix of `words` / 每个前缀之后概率最高的 k 个词."""
    model.eval()
    ids = encode(words)[None]
    probs = model(ids)[0].softmax(-1)
    out = []
    for t in range(len(words) - 1):
        top = probs[t].topk(k)
        out.append({"target": words[t + 1], "p_target": probs[t, STOI[words[t + 1]]].item(),
                    "top": [(VOCAB[i], p) for p, i in zip(top.values.tolist(), top.indices.tolist())]})
    return out

In [ ]:
def main():
    print(f"vocabulary: {len(VOCAB)} words / 词表 {len(VOCAB)} 个词")
    rng = random.Random(0)
    for _ in range(3):
        print("  " + " ".join(sentence(rng)))
    val = encode(make_corpus(2000, seed=999))

    print("\n== 1. loss at initialization / 初始损失 ==")
    torch.manual_seed(0)
    init_loss = evaluate(GPT(len(VOCAB)), val, 32)
    print(f"initial loss {init_loss:.3f} ≈ ln(V) = {math.log(len(VOCAB)):.3f}, perplexity {math.exp(init_loss):.1f} ≈ V")
    assert abs(init_loss - math.log(len(VOCAB))) < 0.3

    print("\n== 2. learning-rate schedules / 学习率调度 ==")
    lrs = [cosine_lr(s, 1000, 1.0, 100) for s in range(1000)]
    assert lrs[0] < 0.02 and abs(max(lrs) - 1.0) < 1e-9 and abs(lrs[-1] - 0.1) < 1e-3
    w = [wsd_lr(s, 1000, 1.0, 100) for s in range(1000)]
    assert w[500] == 1.0 and w[-1] < 0.01
    print("✓ cosine: warms up to the peak, decays to 10%;  WSD: holds the peak, decays only at the end")

    print("\n== 3. little data vs more data / 少量数据 vs 更多数据 ==")
    results = {}
    torch.manual_seed(0)
    print(f"{'untrained':26s} agreement {agreement_accuracy(GPT(len(VOCAB))):.0%}")
    for name, n in [("small (100 sentences)", 100), ("large (20,000 sentences)", 20000)]:
        model, log = train(encode(make_corpus(n, seed=1)), val, steps=600)
        results[name] = (log["train"][-1], log["val"][-1], agreement_accuracy(model))
        print(f"{name:26s} train {log['train'][-1]:.3f}   val {log['val'][-1]:.3f}   "
              f"val perplexity {math.exp(log['val'][-1]):.2f}   agreement {results[name][2]:.0%}")
    small_train, small_val, _ = results["small (100 sentences)"]
    large_train, large_val, large_agree = results["large (20,000 sentences)"]
    assert large_agree > 0.9, "with enough data the model learns agreement across the relative clause"
    assert small_val - small_train > 0.3, "with little data the model memorizes: val >> train"
    assert abs(large_val - large_train) < 0.15, "with enough data train and val stay together"
    assert large_val < small_val
    print("✓ little data: low train loss but high val loss (memorizing);  more data: both low (generalizing)")

### 运行演示 / Run the demo (`pretrain.py`)

In [ ]:
main()

## 动手试试 / Try it yourself

改一改上面任意一个函数或参数，重新运行它所在的单元格，再运行 `main()` 看结果有什么变化。
assert 失败说明你的改动改变了程序检查的性质，这正是值得思考的地方。

Change any function or parameter above, re-run its cell, then run `main()` again. If an assert fails, your change broke a
property the script checks, which is exactly what is worth thinking about.